# Task 4 — Sobel Edge Detection using CUDA across PNG Images
**6CS005 High Performance Computing**

⚠️ **GPU runtime required.** Runtime → Change runtime type → GPU

This notebook:
1. Downloads the stb_image single-file headers (no extra installs)
2. Writes the CUDA source inline
3. Compiles the CUDA programme
4. Generates test PNG images using Python (no external files needed)
5. Runs Sobel edge detection on the GPU
6. Displays input and output images side-by-side

## Step 1 — Verify GPU and download stb_image headers

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
# Download stb_image headers (public domain, single-file, no compile step)
!wget -q https://raw.githubusercontent.com/nothings/stb/master/stb_image.h
!wget -q https://raw.githubusercontent.com/nothings/stb/master/stb_image_write.h
!ls -lh stb_image*.h

## Step 2 — Write the CUDA source file

In [ ]:
%%writefile sobel.cu
/*
 * Task 4: Sobel Edge Detection using CUDA
 * 6CS005 High Performance Computing
 *
 * Compile: nvcc -O2 -o sobel sobel.cu -lm
 * Usage:   ./sobel image1.png [image2.png ...]
 * Output:  edge_<name>.png for each input image
 *
 * Uses stb_image / stb_image_write (public domain headers).
 * Download before compile:
 *   wget https://raw.githubusercontent.com/nothings/stb/master/stb_image.h
 *   wget https://raw.githubusercontent.com/nothings/stb/master/stb_image_write.h
 */
#define STB_IMAGE_IMPLEMENTATION
#include "stb_image.h"
#define STB_IMAGE_WRITE_IMPLEMENTATION
#include "stb_image_write.h"

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { \
    cudaError_t _e=(call); \
    if(_e!=cudaSuccess){fprintf(stderr,"[CUDA ERR] %s at %s:%d\n",cudaGetErrorString(_e),__FILE__,__LINE__);exit(1);} \
} while(0)

#define TILE_W 16
#define TILE_H 16

/*
 * Sobel CUDA kernel — each thread handles ONE pixel.
 * Gx = [[-1,0,1],[-2,0,2],[-1,0,1]]
 * Gy = [[-1,-2,-1],[0,0,0],[1,2,1]]
 * Zero-padding at borders.
 */
__global__ void sobel_kernel(const unsigned char *d_in, unsigned char *d_out,
                             int width, int height) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if(x>=width || y>=height) return;

    /* inline zero-padded pixel read */
    #define PX(row,col) \
        (((row)<0||(row)>=height||(col)<0||(col)>=width) ? 0 : \
         (int)(unsigned char)d_in[(row)*width+(col)])

    int gx = -PX(y-1,x-1) + PX(y-1,x+1)
             -2*PX(y,x-1) + 2*PX(y,x+1)
             -PX(y+1,x-1) + PX(y+1,x+1);

    int gy = -PX(y-1,x-1) - 2*PX(y-1,x) - PX(y-1,x+1)
             +PX(y+1,x-1) + 2*PX(y+1,x) + PX(y+1,x+1);

    int mag = (int)sqrtf((float)(gx*gx + gy*gy));
    if(mag>255) mag=255;
    if(mag<0)   mag=0;
    d_out[y*width+x] = (unsigned char)mag;
    #undef PX
}

static void process_image(const char *inpath) {
    /* 1. Load image to CPU */
    int w,h,ch;
    unsigned char *h_rgb = stbi_load(inpath,&w,&h,&ch,0);
    if(!h_rgb){fprintf(stderr,"[ERROR] Cannot load '%s': %s\n",inpath,stbi_failure_reason());return;}
    printf("[INFO] Loaded '%s'  %dx%d  channels=%d\n",inpath,w,h,ch);
    size_t np=(size_t)w*h;

    /* 2. Convert RGB→Grayscale on CPU */
    unsigned char *h_gray=(unsigned char*)malloc(np);
    for(size_t i=0;i<np;i++){
        if(ch>=3) h_gray[i]=(unsigned char)(0.299f*h_rgb[i*ch+0]+0.587f*h_rgb[i*ch+1]+0.114f*h_rgb[i*ch+2]);
        else      h_gray[i]=h_rgb[i*ch];
    }
    stbi_image_free(h_rgb);

    /* 3. Allocate GPU memory */
    unsigned char *d_in=NULL,*d_out=NULL;
    CUDA_CHECK(cudaMalloc((void**)&d_in, np));
    CUDA_CHECK(cudaMalloc((void**)&d_out,np));

    /* 4. Transfer to GPU */
    CUDA_CHECK(cudaMemcpy(d_in,h_gray,np,cudaMemcpyHostToDevice));

    /* 5. Launch kernel */
    dim3 thr(TILE_W,TILE_H);
    dim3 blk((w+TILE_W-1)/TILE_W,(h+TILE_H-1)/TILE_H);
    printf("[INFO] Kernel: %dx%d blocks, %dx%d threads\n",blk.x,blk.y,thr.x,thr.y);
    sobel_kernel<<<blk,thr>>>(d_in,d_out,w,h);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    /* 6. Transfer back to CPU */
    unsigned char *h_edge=(unsigned char*)malloc(np);
    CUDA_CHECK(cudaMemcpy(h_edge,d_out,np,cudaMemcpyDeviceToHost));

    /* 7. Build output filename */
    const char *base=strrchr(inpath,'/');
    if(!base)base=strrchr(inpath,'\\');
    base=base?base+1:inpath;
    char outpath[1024];
    snprintf(outpath,sizeof(outpath),"edge_%s",base);
    size_t olen=strlen(outpath);
    if(olen<4||strcmp(outpath+olen-4,".png")!=0)
        snprintf(outpath+olen,sizeof(outpath)-olen,".png");

    /* 8. Save output PNG */
    if(stbi_write_png(outpath,w,h,1,h_edge,w))
        printf("[INFO] Edge image saved: '%s'\n",outpath);
    else
        fprintf(stderr,"[ERROR] Failed to write '%s'\n",outpath);

    /* 9. Free memory */
    CUDA_CHECK(cudaFree(d_in)); CUDA_CHECK(cudaFree(d_out));
    free(h_gray); free(h_edge);
}

int main(int argc,char*argv[]){
    if(argc<2){fprintf(stderr,"Usage: %s image1.png [image2.png ...]\n",argv[0]);return 1;}
    int dev; cudaGetDevice(&dev);
    cudaDeviceProp p; cudaGetDeviceProperties(&p,dev);
    printf("[INFO] GPU: %s (SM %d.%d)\n",p.name,p.major,p.minor);
    for(int i=1;i<argc;i++) process_image(argv[i]);
    printf("[INFO] All done.\n");
    return 0;
}

## Step 3 — Compile

In [ ]:
!nvcc -O2 -o sobel sobel.cu -lm && echo '✅ CUDA compilation successful'

## Step 4 — Generate test PNG images using Python (no external files needed)

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np

def make_test_image(name, width=256, height=256):
    """Create a synthetic test PNG with shapes/gradients to produce visible edges."""
    img = Image.new('RGB', (width, height), color=(40, 40, 40))
    draw = ImageDraw.Draw(img)
    # White rectangle
    draw.rectangle([30, 30, 120, 120], fill=(220, 220, 220))
    # White circle
    draw.ellipse([140, 40, 230, 130], fill=(200, 200, 200))
    # Diagonal lines
    for i in range(0, width, 20):
        draw.line([(i, height//2), (i+20, height)], fill=(180,180,180), width=2)
    # Inner dark rectangle
    draw.rectangle([50, 150, 200, 220], fill=(15, 15, 15), outline=(255,255,255), width=2)
    img.save(name)
    print(f'Created {name}  {width}x{height}')
    return img

# Create a real-world-like test: a checkerboard
def make_checkerboard(name, size=256, squares=8):
    sq = size // squares
    arr = np.zeros((size, size, 3), dtype=np.uint8)
    for r in range(squares):
        for c in range(squares):
            val = 240 if (r + c) % 2 == 0 else 20
            arr[r*sq:(r+1)*sq, c*sq:(c+1)*sq] = val
    Image.fromarray(arr).save(name)
    print(f'Created {name}  {size}x{size}')

img1 = make_test_image('test_shapes.png')
make_checkerboard('test_checker.png')

# Also download a real image for a richer test
import urllib.request
try:
    url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/5/50/Vd-Orig.png/256px-Vd-Orig.png'
    urllib.request.urlretrieve(url, 'test_real.png')
    print('Downloaded test_real.png from Wikipedia')
except Exception as e:
    print(f'Download failed ({e}), using only generated images')

## Step 5 — Run Sobel edge detection on all test images

In [ ]:
import glob
test_images = [f for f in glob.glob('test_*.png')]
print('Processing:', test_images)
!./sobel test_shapes.png test_checker.png $(ls test_real.png 2>/dev/null)

## Step 6 — Display results side-by-side

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

pairs = []
for orig in sorted(glob.glob('test_*.png')):
    edge = 'edge_' + orig
    if os.path.exists(edge):
        pairs.append((orig, edge))

if not pairs:
    print('No edge images found — check compilation and run steps above.')
else:
    fig, axes = plt.subplots(len(pairs), 2, figsize=(10, 5 * len(pairs)))
    if len(pairs) == 1:
        axes = [axes]
    for i, (orig, edge) in enumerate(pairs):
        o_img = mpimg.imread(orig)
        e_img = mpimg.imread(edge)
        axes[i][0].imshow(o_img, cmap='gray' if o_img.ndim==2 else None)
        axes[i][0].set_title(f'Input: {orig}', fontsize=11)
        axes[i][0].axis('off')
        axes[i][1].imshow(e_img, cmap='gray')
        axes[i][1].set_title(f'Sobel Edge: {edge}', fontsize=11)
        axes[i][1].axis('off')
    plt.suptitle('Sobel Edge Detection — CUDA GPU Results', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig('sobel_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Results saved to sobel_results.png')

## Step 7 — Timing comparison (single image, for report)
This cell measures GPU runtime using CUDA events.

In [ ]:
%%writefile sobel_timed.cu
/* Timed version — adds cudaEvent timing around the kernel */
#define STB_IMAGE_IMPLEMENTATION
#include "stb_image.h"
#define STB_IMAGE_WRITE_IMPLEMENTATION
#include "stb_image_write.h"
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#define CUDA_CHECK(c) do{cudaError_t e=(c);if(e!=cudaSuccess){fprintf(stderr,"%s\n",cudaGetErrorString(e));exit(1);}}while(0)
#define TILE 16
__global__ void sobel_k(const unsigned char*in,unsigned char*out,int w,int h){
    int x=blockIdx.x*blockDim.x+threadIdx.x, y=blockIdx.y*blockDim.y+threadIdx.y;
    if(x>=w||y>=h)return;
    #define P(r,c) (((r)<0||(r)>=h||(c)<0||(c)>=w)?0:(int)(unsigned char)in[(r)*w+(c)])
    int gx=-P(y-1,x-1)+P(y-1,x+1)-2*P(y,x-1)+2*P(y,x+1)-P(y+1,x-1)+P(y+1,x+1);
    int gy=-P(y-1,x-1)-2*P(y-1,x)-P(y-1,x+1)+P(y+1,x-1)+2*P(y+1,x)+P(y+1,x+1);
    int m=(int)sqrtf((float)(gx*gx+gy*gy)); if(m>255)m=255; out[y*w+x]=(unsigned char)m;
    #undef P
}
int main(int argc,char*argv[]){
    if(argc<2){fprintf(stderr,"Usage: sobel_timed <image>\n");return 1;}
    int w,h,ch; unsigned char*rgb=stbi_load(argv[1],&w,&h,&ch,0);
    if(!rgb){fprintf(stderr,"Load failed\n");return 1;}
    size_t np=(size_t)w*h;
    unsigned char*gray=(unsigned char*)malloc(np);
    for(size_t i=0;i<np;i++) gray[i]=(ch>=3)?(unsigned char)(.299f*rgb[i*ch]+.587f*rgb[i*ch+1]+.114f*rgb[i*ch+2]):rgb[i*ch];
    stbi_image_free(rgb);
    unsigned char*din,*dout;
    CUDA_CHECK(cudaMalloc((void**)&din,np)); CUDA_CHECK(cudaMalloc((void**)&dout,np));
    CUDA_CHECK(cudaMemcpy(din,gray,np,cudaMemcpyHostToDevice));
    cudaEvent_t start,stop; float ms;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    dim3 thr(TILE,TILE),blk((w+TILE-1)/TILE,(h+TILE-1)/TILE);
    cudaEventRecord(start);
    sobel_k<<<blk,thr>>>(din,dout,w,h);
    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms,start,stop);
    printf("[TIMING] Image %dx%d  kernel time: %.3f ms\n",w,h,ms);
    unsigned char*edge=(unsigned char*)malloc(np);
    CUDA_CHECK(cudaMemcpy(edge,dout,np,cudaMemcpyDeviceToHost));
    stbi_write_png("edge_timed.png",w,h,1,edge,w);
    printf("[INFO] Saved edge_timed.png\n");
    cudaFree(din);cudaFree(dout);free(gray);free(edge);
    return 0;
}

In [ ]:
!nvcc -O2 -o sobel_timed sobel_timed.cu -lm && ./sobel_timed test_shapes.png